# Multimodal Crop Health Monitoring — Setup & EDA

**Author:** Khang Phan  
**Purpose:** Environment verification, dataset loading, and exploratory data analysis.

This notebook:
1. Checks that all dependencies are installed
2. Loads the real crop yield CSVs from `../data/raw/`
3. Produces summary statistics and visualizations

## 1. Environment Check

In [ ]:
import importlib.metadata, sys

# Maps pip package name → import name (for display)
packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "xgboost": "xgboost",
    "torch": "torch",
    "torchvision": "torchvision",
    "opencv-python": "cv2",
    "streamlit": "streamlit",
    "seaborn": "seaborn",
    "pillow": "PIL",
    "kaggle": "kaggle",
}

all_ok = True
for pip_name, import_name in packages.items():
    try:
        version = importlib.metadata.version(pip_name)
        print(f"  [OK]      {pip_name:<20} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  [MISSING] {pip_name:<20} → pip install {pip_name}")
        all_ok = False

print(f"\nPython {sys.version}")
print("\nEnvironment ready!" if all_ok else "\nInstall missing packages, then re-run.")

## 2. Download Yield Dataset (Kaggle API)

**Setup:**
- **Google Colab:** Upload `kaggle.json` to the `/content/` folder (left sidebar → Files → Upload), then run the cell below
- **Local:** Place `kaggle.json` at `~/.kaggle/kaggle.json`

Get your `kaggle.json` at: https://www.kaggle.com/settings → API → **Create New Token**

In [ ]:
import os, json
from pathlib import Path

DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

kaggle_dir              = Path.home() / ".kaggle"
kaggle_json_path        = kaggle_dir / "kaggle.json"
content_kaggle_json_path = Path("/content/kaggle.json")

if not kaggle_json_path.exists():
    if content_kaggle_json_path.exists():
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        content_kaggle_json_path.rename(kaggle_json_path)
        print(f"Moved {content_kaggle_json_path} to {kaggle_json_path}")
    else:
        raise FileNotFoundError(f"kaggle.json not found. Upload it to /content/ or place it at {kaggle_json_path}")

with open(kaggle_json_path, "r") as f:
    creds = json.load(f)
os.environ["KAGGLE_USERNAME"] = creds["username"]
os.environ["KAGGLE_KEY"]      = creds["key"]

import kaggle
kaggle.api.authenticate()
print("Downloading crop yield dataset...")
kaggle.api.dataset_download_files(
    "patelris/crop-yield-prediction-dataset",
    path=str(DATA_RAW),
    unzip=True,
    quiet=False,
)

print("\nFiles in data/raw/:")
for f in sorted(DATA_RAW.glob("*.csv")):
    print(f"  {f.name}")

## 2b. Download All PlantVillage Datasets & Merge

Downloads from all three sources and merges into one unified `data/raw/PlantVillage_combined/` folder:

| Source | Dataset |
|---|---|
| Kaggle | `abdallahalidev/plantvillage-dataset` (~54k images, color/gray/segmented) |
| Kaggle | `emmarex/plantdisease` (~54k images) |
| GitHub | `gabrieldgf4/PlantVillage-Dataset` |

All are merged by class folder — duplicates skipped by filename.

In [ ]:
import kaggle, os, shutil, subprocess, sys
from pathlib import Path

DATA_RAW = Path("../data/raw")
COMBINED = DATA_RAW / "PlantVillage_combined"
COMBINED.mkdir(parents=True, exist_ok=True)

kaggle.api.authenticate()


def merge_into_combined(source_dir: Path, label: str):
    """Copy all class-folder images from source_dir into COMBINED, skip duplicates."""
    if not source_dir.exists():
        print(f"  [SKIP] {label} — folder not found at {source_dir}")
        return 0
    copied = 0
    for class_dir in source_dir.iterdir():
        if not class_dir.is_dir():
            continue
        dest = COMBINED / class_dir.name
        dest.mkdir(exist_ok=True)
        for img in class_dir.glob("*"):
            if img.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                target = dest / img.name
                if not target.exists():
                    shutil.copy2(img, target)
                    copied += 1
    print(f"  [{label}] +{copied:,} new images")
    return copied


# ── 1. abdallahalidev/plantvillage-dataset ───────────────────────────────
print("1. Downloading abdallahalidev/plantvillage-dataset (~4.4 GB)...")
dl_dir = DATA_RAW / "abdallahalidev"
dl_dir.mkdir(exist_ok=True)
kaggle.api.dataset_download_files(
    "abdallahalidev/plantvillage-dataset",
    path=str(dl_dir), unzip=True, quiet=False,
)
# Find color subfolder
color_candidates = [dl_dir / "plantvillage dataset" / "color",
                    dl_dir / "color", dl_dir / "plantvillagedataset" / "color"]
color_dir = next((p for p in color_candidates if p.exists()), None)
if color_dir:
    merge_into_combined(color_dir, "abdallahalidev/color")
else:
    print(f"  Could not find color folder, contents: {list(dl_dir.rglob('*'))[:10]}")


# ── 2. emmarex/plantdisease ──────────────────────────────────────────────
print("\n2. Downloading emmarex/plantdisease...")
dl_dir2 = DATA_RAW / "emmarex"
dl_dir2.mkdir(exist_ok=True)
kaggle.api.dataset_download_files(
    "emmarex/plantdisease",
    path=str(dl_dir2), unzip=True, quiet=False,
)
pv_candidates = [dl_dir2 / "PlantVillage", dl_dir2]
pv_dir = next((p for p in pv_candidates if p.exists() and any(p.iterdir())), None)
if pv_dir:
    merge_into_combined(pv_dir, "emmarex/plantdisease")


# ── 3. gabrieldgf4/PlantVillage-Dataset (GitHub) ────────────────────────
print("\n3. Downloading gabrieldgf4/PlantVillage-Dataset from GitHub...")
gh_dir = DATA_RAW / "gabrieldgf4"
if gh_dir.exists():
    shutil.rmtree(gh_dir)
result = subprocess.run(
    ["git", "clone", "--depth=1",
     "https://github.com/gabrieldgf4/PlantVillage-Dataset.git",
     str(gh_dir)],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)
# Find image folders inside repo
gh_candidates = [gh_dir / "dataset" / "color", gh_dir / "color",
                 gh_dir / "dataset", gh_dir]
gh_img_dir = next((p for p in gh_candidates
                   if p.exists() and any(p.iterdir())), None)
if gh_img_dir:
    merge_into_combined(gh_img_dir, "gabrieldgf4/GitHub")


# ── Summary ──────────────────────────────────────────────────────────────
print("\n=== Combined Dataset Summary ===")
class_folders = [d for d in COMBINED.iterdir() if d.is_dir()]
total = sum(len(list(d.glob("*.jpg")) + list(d.glob("*.JPG")) + list(d.glob("*.png")))
            for d in class_folders)
print(f"Classes : {len(class_folders)}")
print(f"Images  : {total:,}")
print(f"Path    : {COMBINED}")
print("\nClasses found:")
for d in sorted(class_folders):
    n = len(list(d.glob("*")))
    print(f"  {d.name:<50} {n:>5} images")

## 3. Crop Yield Dataset — Load & Explore

Loading `Agri_yield_prediction.csv` — field-level dataset with soil chemistry, weather, irrigation, and crop type.
Weather features (Temperature, Humidity, Rainfall) align with the UI sliders so no separate weather input is needed.
Yield target is in **t/ha** (tons per hectare).

In [ ]:
import pandas as pd
from pathlib import Path

DATA_RAW = Path('../data/raw')
csv_path = DATA_RAW / 'Agri_yield_prediction.csv'

if not csv_path.exists():
    raise FileNotFoundError(
        'Agri_yield_prediction.csv not found. Place it in data/raw/ or download from Kaggle.'
    )

df = pd.read_csv(csv_path)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Crop types: {df["Crop_Type"].unique()}')
print(f'Regions   : {df["Region"].unique()}')
print(f'Yield range: {df["Yield"].min():.2f} – {df["Yield"].max():.2f} t/ha')
df.head()

In [ ]:
print("Summary statistics for numeric columns:")
df.describe().round(2)

## 4. Yield Dataset — Visualizations

In [ ]:
import matplotlib.pyplot as plt
import pathlib

RESULTS = pathlib.Path("../results")
RESULTS.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Crop Yield Dataset — Exploratory Analysis", fontsize=15, fontweight="bold")

# 1. Yield distribution
axes[0, 0].hist(df["hg/ha_yield"], bins=40, color="#4CAF50", edgecolor="white", alpha=0.85)
axes[0, 0].set_title("Crop Yield Distribution")
axes[0, 0].set_xlabel("Yield (hg/ha)")
axes[0, 0].set_ylabel("Count")

# 2. Average yield by crop
if "Item" in df.columns:
    crop_yield = df.groupby("Item")["hg/ha_yield"].mean().sort_values(ascending=False)
    axes[0, 1].barh(crop_yield.index, crop_yield.values, color="#2196F3", alpha=0.85)
    axes[0, 1].set_title("Average Yield by Crop")
    axes[0, 1].set_xlabel("Mean Yield (hg/ha)")

# 3. Rainfall vs Yield scatter
axes[1, 0].scatter(df["average_rain_fall_mm_per_year"], df["hg/ha_yield"],
                   alpha=0.3, s=12, color="#FF9800")
axes[1, 0].set_title("Rainfall vs Yield")
axes[1, 0].set_xlabel("Avg Rainfall (mm/year)")
axes[1, 0].set_ylabel("Yield (hg/ha)")

# 4. Temperature vs Yield scatter
axes[1, 1].scatter(df["avg_temp"], df["hg/ha_yield"],
                   alpha=0.3, s=12, color="#9C27B0")
axes[1, 1].set_title("Temperature vs Yield")
axes[1, 1].set_xlabel("Avg Temperature (°C)")
axes[1, 1].set_ylabel("Yield (hg/ha)")

plt.tight_layout()
plt.savefig(RESULTS / "eda_yield.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved to results/eda_yield.png")

## 5. PlantVillage Image Dataset — Exploration

Counts images per class. Falls back to generating synthetic sample images if dataset is not yet downloaded.

In [ ]:
import pathlib, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from PIL import Image

pv_dir = pathlib.Path("../data/raw/PlantVillage")

if pv_dir.exists():
    # Real dataset: count images per class folder
    class_counts = {d.name: len(list(d.glob("*.jpg")) + list(d.glob("*.JPG")))
                    for d in sorted(pv_dir.iterdir()) if d.is_dir()}
    use_synthetic = False
    print(f"PlantVillage: {len(class_counts)} classes, {sum(class_counts.values()):,} total images")
else:
    # Synthetic class distribution mimicking PlantVillage
    use_synthetic = True
    print("PlantVillage not found — using representative synthetic class distribution.\n")
    class_counts = {
        "Tomato_healthy": 1591, "Tomato_Early_blight": 1000, "Tomato_Late_blight": 1909,
        "Tomato_Leaf_Mold": 952,  "Tomato_Septoria_leaf_spot": 1771,
        "Potato_healthy": 152,   "Potato_Early_blight": 1000, "Potato_Late_blight": 1000,
        "Corn_healthy": 1162,    "Corn_Common_rust": 1192,    "Corn_Northern_Leaf_Blight": 985,
        "Apple_healthy": 1645,   "Apple_scab": 630,           "Apple_Black_rot": 621,
        "Grape_healthy": 423,    "Grape_Black_rot": 1180,     "Grape_Leaf_blight": 1076,
        "Pepper_healthy": 1478,  "Pepper_Bacterial_spot": 997,
    }

# Plot class distribution (top 15)
top = dict(sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:15])
colors = ["#4CAF50" if "healthy" in k.lower() else "#F44336" for k in top]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(list(top.keys()), list(top.values()), color=colors, alpha=0.85, edgecolor="white")
ax.set_xlabel("Number of Images")
ax.set_title("PlantVillage — Top 15 Classes by Image Count\n(green = healthy, red = disease)",
             fontweight="bold")
ax.bar_label(bars, padding=3, fontsize=8)
plt.tight_layout()
plt.savefig("../results/eda_plantvillage.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Total classes shown: {len(top)} | Plot saved to results/eda_plantvillage.png")

## 6. Sample Image Grid (PlantVillage)

Displays a 3×3 grid of sample leaf images — real if available, otherwise synthetic color patches for layout verification.

In [ ]:
import random, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

pv_dir = Path("../data/raw/PlantVillage")
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("Sample Leaf Images — PlantVillage Dataset", fontsize=13, fontweight="bold")

all_images = list(pv_dir.rglob("*.jpg")) + list(pv_dir.rglob("*.JPG")) if pv_dir.exists() else []

for i, ax in enumerate(axes.flat):
    if all_images:
        img_path = random.choice(all_images)
        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
        label = img_path.parent.name.replace("_", "\n")
    else:
        # Synthetic placeholder: colored square with label
        color = np.random.uniform(0.1, 0.9, (224, 224, 3))
        color[:, :, 1] = color[:, :, 1] * 0.6 + 0.2  # greenish tint
        ax.imshow(color)
        labels = ["Tomato_healthy", "Tomato_Early_blight", "Potato_Late_blight",
                  "Corn_healthy", "Apple_scab", "Grape_Black_rot",
                  "Pepper_healthy", "Corn_Common_rust", "Potato_healthy"]
        label = labels[i].replace("_", "\n")
    ax.set_title(label, fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.savefig("../results/sample_images.png", dpi=100, bbox_inches="tight")
plt.show()
note = "(synthetic placeholders)" if not all_images else "(real PlantVillage images)"
print(f"Image grid {note} saved to results/sample_images.png")

## 7. Correlation Heatmap — Yield Features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_cols = ["hg/ha_yield", "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "Year"]
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels([c.replace("_", "\n") for c in num_cols], fontsize=8)
ax.set_yticklabels([c.replace("_", "\n") for c in num_cols], fontsize=8)

for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)

ax.set_title("Feature Correlation Matrix — Crop Yield Data", fontweight="bold")
plt.tight_layout()
plt.savefig("../results/correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print("Heatmap saved to results/correlation_heatmap.png")

## 8. Setup Complete

All outputs saved to `../results/`. Summary:

| Output | Description |
|---|---|
| `eda_yield.png` | Yield distribution, average by crop, rainfall/temp scatter |
| `eda_plantvillage.png` | PlantVillage class distribution (runs when image data is present) |
| `sample_images.png` | 3×3 leaf image grid (runs when image data is present) |
| `correlation_heatmap.png` | Feature correlation matrix |

**Next steps:**
- Run `src/yield_predictor.py` to train the XGBoost yield model
- Download PlantVillage images, then run `src/disease_classifier.py`
- Launch the dashboard: `streamlit run ui/app.py`

---

## 9. Yield Prediction — Data Preprocessing

Encode categorical columns and split into train/test sets.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv(Path('../data/raw/Agri_yield_prediction.csv'))

CATEGORICAL = ['Crop_Type', 'Soil_Type', 'Fertilizer_Type', 'Pesticide_Usage']
NUMERIC     = ['Temperature', 'Humidity', 'Rainfall',
               'pH', 'N', 'P', 'K', 'Irrigation_Frequency']
TARGET      = 'Yield'

encoders = {}
for col in CATEGORICAL:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

FEATURES = NUMERIC + [f'{c}_enc' for c in CATEGORICAL]
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')
print(f'Features: {FEATURES}')
print(f'Target  : {TARGET} (t/ha)')
X_train.head()

## 10. Train Baseline — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_r2  = r2_score(y_test, rf_preds)

print(f"Random Forest")
print(f"  MAE : {rf_mae:,.0f} hg/ha")
print(f"  R²  : {rf_r2:.4f}")

## 11. Train Main Model — XGBoost

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_r2  = r2_score(y_test, xgb_preds)
print(f'XGBoost  MAE={xgb_mae:.4f} t/ha  R²={xgb_r2:.4f}')

## 12. Evaluation — Predicted vs Actual & Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("XGBoost Yield Predictor — Evaluation", fontsize=13, fontweight="bold")

# 1. Predicted vs Actual
axes[0].scatter(y_test, xgb_preds, alpha=0.3, s=10, color="#2196F3")
lims = [min(y_test.min(), xgb_preds.min()), max(y_test.max(), xgb_preds.max())]
axes[0].plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual Yield (t/ha)")
axes[0].set_ylabel("Predicted Yield (t/ha)")
axes[0].set_title(f"Predicted vs Actual  (R² = {xgb_r2:.3f})")
axes[0].legend()

# 2. Feature importance
importances = xgb_model.feature_importances_
feat_names  = FEATURES
order = np.argsort(importances)
axes[1].barh([feat_names[i] for i in order], importances[order], color="#4CAF50", alpha=0.85)
axes[1].set_xlabel("Importance Score")
axes[1].set_title("Feature Importance")

plt.tight_layout()
plt.savefig(RESULTS / "yield_model_evaluation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved to results/yield_model_evaluation.png")


## 13. Save Model

In [ ]:
import pickle

save_path = RESULTS / 'yield_model.pkl'
with open(save_path, 'wb') as f:
    pickle.dump({
        'model': xgb_model,
        'encoders': encoders,
        'features': FEATURES,
    }, f)

print(f'Model saved to {save_path}')
print(f'  MAE : {xgb_mae:.4f} t/ha')
print(f'  R²  : {xgb_r2:.4f}')

---

## 14. Disease Classifier — Data Loading & Preprocessing

Loads PlantVillage images using PyTorch's `ImageFolder`.  
> **Tip:** Enable GPU in Colab → Runtime → Change runtime type → T4 GPU for much faster training.

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

DATA_RAW = Path("../data/raw")

# Priority: combined > abdallahalidev color > emmarex > old PlantVillage
candidates = [
    DATA_RAW / "PlantVillage_combined",
    DATA_RAW / "abdallahalidev" / "plantvillage dataset" / "color",
    DATA_RAW / "abdallahalidev" / "color",
    DATA_RAW / "emmarex" / "PlantVillage",
    DATA_RAW / "PlantVillage",
    DATA_RAW / "color",
]
PV_DIR = next((p for p in candidates if p.exists() and any(p.iterdir())), None)
if PV_DIR is None:
    raise FileNotFoundError("No PlantVillage folder found. Run Section 2b first.")
print(f"Using dataset: {PV_DIR}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(PV_DIR, transform=train_transform)
CLASS_NAMES  = full_dataset.classes
NUM_CLASSES  = len(CLASS_NAMES)

n_val   = int(0.2 * len(full_dataset))
n_train = len(full_dataset) - n_val
train_ds, val_ds = random_split(full_dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
val_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Classes : {NUM_CLASSES}")
print(f"Train   : {n_train:,} images")
print(f"Val     : {n_val:,} images")
print(f"\nAll classes:")
for c in CLASS_NAMES:
    print(f"  {c}")

## 15. Build Model — ResNet-18 with Transfer Learning

In [ ]:
import torch.nn as nn
from torchvision import models

# Load pretrained ResNet-18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all layers — only train the final classifier head
for param in model.parameters():
    param.requires_grad = False

# Replace final layer with our number of classes
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
print(f"Model ready on: {DEVICE}")

## 16. Train Disease Classifier

In [ ]:
EPOCHS = 5
history = {"train_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc  = correct / total

    # ── Validation ────────────────────────────────────────────────────────
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total

    scheduler.step()
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{EPOCHS}  loss={train_loss:.4f}  "
          f"train_acc={train_acc:.3f}  val_acc={val_acc:.3f}")

## 17. Training Curves & Save Model

In [ ]:
import matplotlib.pyplot as plt
import torch
from pathlib import Path

RESULTS = Path("../results")
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Disease Classifier — Training Results", fontweight="bold")

axes[0].plot(epochs_range, history["train_loss"], marker="o", color="#F44336", label="Train Loss")
axes[0].set_title("Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], marker="o", color="#2196F3", label="Train Acc")
axes[1].plot(epochs_range, history["val_acc"],   marker="s", color="#4CAF50", label="Val Acc")
axes[1].set_title("Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS / "disease_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

# Save model
save_path = RESULTS / "disease_model.pth"
torch.save({"model_state": model.state_dict(), "classes": CLASS_NAMES}, save_path)
print(f"Model saved to {save_path}")
print(f"Final val accuracy: {history['val_acc'][-1]:.3f}")